In [1]:
import pandas as pd

df_train = pd.read_csv("data/claims_train_clean.csv")
df_test  = pd.read_csv("data/claims_test_clean.csv")

print(df_train.shape, df_test.shape)
df_train.head()


(541416, 13) (135373, 13)


,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region,ClaimRate
0,2122523.0,0,0.43,D,7,18,36,95,B1,Regular,1054,R24,0.0
1,3173420.0,0,0.10,D,7,17,80,95,B2,Regular,598,R25,0.0
2,1188619.0,0,0.33,E,7,3,36,76,B6,Regular,4172,R82,0.0
3,31400.0,0,0.56,A,5,4,73,52,B13,Diesel,15,R24,0.0
4,3138755.0,0,0.27,E,8,0,37,50,B11,Diesel,3021,R53,0.0


In [2]:
drop_cols = ["IDpol", "ClaimRate", "VehBrand", "VehGas", "Region"]

df_train = df_train.drop(columns=drop_cols)
df_test  = df_test.drop(columns=drop_cols)


In [8]:
#mapping area
area_labels = sorted(df_train["Area"].unique())
area_to_int = {a: i for i, a in enumerate(area_labels)}
print(area_to_int)

df_train["Area_enc"] = df_train["Area"].map(area_to_int).astype(int)
df_test["Area_enc"] = df_test["Area"].map(area_to_int).fillna(-1).astype(int)

{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5}


In [9]:
#Used features
numeric_features = ["Exposure", "VehPower", "VehAge", "DrivAge", "BonusMalus", "Density"]
categorical_features = ["Area_enc"]

feature_cols = numeric_features + categorical_features

X_train = df_train[feature_cols].copy()
X_test  = df_test[feature_cols].copy()

y_train = df_train["ClaimNb"].values
y_test  = df_test["ClaimNb"].values


In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

# Scale only numeric columns
X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features]  = scaler.transform(X_test[numeric_features])


In [12]:
import numpy as np
offset_train = np.log(df_train["Exposure"].values + 1e-10)
offset_test  = np.log(df_test["Exposure"].values + 1e-10)

In [13]:
X_train_final = X_train_scaled.values.astype(float)
X_test_final  = X_test_scaled.values.astype(float)

print("X_train:", X_train_final.shape)
print("X_test:", X_test_final.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (541416, 7)
X_test: (135373, 7)
y_train: (541416,)
y_test: (135373,)


In [15]:
from sklearn.linear_model import PoissonRegressor
import numpy as np

# avoid division by zero: use only rows with Exposure > 0
train_mask = df_train["Exposure"] > 0
test_mask  = df_test["Exposure"] > 0

X_train_poi = X_train_final[train_mask]
X_test_poi  = X_test_final[test_mask]

y_train_claims = df_train.loc[train_mask, "ClaimNb"].values
y_test_claims  = df_test.loc[test_mask, "ClaimNb"].values

exp_train = df_train.loc[train_mask, "Exposure"].values
exp_test  = df_test.loc[test_mask, "Exposure"].values

# model the *rate* = claims per unit exposure
y_train_rate = y_train_claims / exp_train

print("min rate:", y_train_rate.min(), "max rate:", y_train_rate.max())

# Poisson regression on rate, weighted by exposure
poisson = PoissonRegressor(alpha=1e-4, max_iter=300)

poisson.fit(X_train_poi, y_train_rate, sample_weight=exp_train)

# predictions: first predict rate, then multiply by exposure to get expected claims
pred_rate_train = poisson.predict(X_train_poi)
pred_rate_test  = poisson.predict(X_test_poi)

pred_claims_train = pred_rate_train * exp_train
pred_claims_test  = pred_rate_test * exp_test

# RMSE in terms of predicted ClaimNb
rmse_train_poi = np.sqrt(np.mean((y_train_claims - pred_claims_train) ** 2))
rmse_test_poi  = np.sqrt(np.mean((y_test_claims  - pred_claims_test)  ** 2))

print("Poisson train RMSE:", rmse_train_poi)
print("Poisson test RMSE:", rmse_test_poi)


min rate: 0.0 max rate: 732.0000000000188
Poisson train RMSE: 0.2374797336163933
Poisson test RMSE: 0.24394322285600015


c:\ProgramData\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\ProgramData\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\ProgramData\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\ProgramData\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\ProgramData\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

In [16]:
for name, coef in zip(feature_cols, poisson.coef_):
    print(f"{name}: {coef:.4f}")


Exposure: -0.4683
VehPower: 0.0107
VehAge: -0.1722
DrivAge: 0.1386
BonusMalus: 0.3006
Density: -0.0007
Area_enc: 0.0330


In [17]:
print("Intercept:", poisson.intercept_)


Intercept: -2.1542717165487137


In [ ]:
#Same model but with all categorical values encoded
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import PoissonRegressor

df_train = pd.read_csv("data/claims_train_clean.csv")
df_test  = pd.read_csv("data/claims_test_clean.csv")

print("train shape:", df_train.shape)
print("test shape:", df_test.shape)
print(df_train.head(3))

df_train = df_train.drop(columns=["IDpol", "ClaimRate"])
df_test  = df_test.drop(columns=["IDpol", "ClaimRate"])

numeric_features = ["Exposure", "VehPower", "VehAge", "DrivAge", "BonusMalus", "Density"]
categorical_features = ["Area", "VehBrand", "VehGas", "Region"]

#one-hot encode features
train_cat_dummies = pd.get_dummies(df_train[categorical_features], drop_first=False)
test_cat_dummies  = pd.get_dummies(df_test[categorical_features], drop_first=False)

print("categorical dummy shape (train):", train_cat_dummies.shape)
print("categorical dummy shape (test):", test_cat_dummies.shape)

X_train_num = df_train[numeric_features].copy()
X_test_num  = df_test[numeric_features].copy()

# concatenate numeric + dummies
X_train_full = pd.concat([X_train_num.reset_index(drop=True),
                          train_cat_dummies.reset_index(drop=True)], axis=1)

X_test_full = pd.concat([X_test_num.reset_index(drop=True),
                         test_cat_dummies.reset_index(drop=True)], axis=1)

feature_cols = list(X_train_full.columns)

print("X_train_full shape:", X_train_full.shape)
print("X_test_full shape:", X_test_full.shape)

# targets
y_train = df_train["ClaimNb"].values
y_test  = df_test["ClaimNb"].values

# ---------------------------
# scale numeric features only
# ---------------------------
scaler = StandardScaler()

X_train_scaled = X_train_full.copy()
X_test_scaled  = X_test_full.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(X_train_full[numeric_features])
X_test_scaled[numeric_features]  = scaler.transform(X_test_full[numeric_features])

# final matrices as numpy arrays
X_train_final = X_train_scaled.values.astype(float)
X_test_final  = X_test_scaled.values.astype(float)

print("X_train_final shape:", X_train_final.shape)
print("X_test_final shape:", X_test_final.shape)

# ---------------------------
# prepare Poisson target and weights (rate + exposure)
# ---------------------------
train_mask = df_train["Exposure"] > 0
test_mask  = df_test["Exposure"] > 0

X_train_poi = X_train_final[train_mask]
X_test_poi  = X_test_final[test_mask]

y_train_claims = df_train.loc[train_mask, "ClaimNb"].values
y_test_claims  = df_test.loc[test_mask, "ClaimNb"].values

exp_train = df_train.loc[train_mask, "Exposure"].values
exp_test  = df_test.loc[test_mask, "Exposure"].values

y_train_rate = y_train_claims / exp_train

print("rate min:", y_train_rate.min(), "rate max:", y_train_rate.max())

# ---------------------------
# fit Poisson regression
# ---------------------------
poisson = PoissonRegressor(alpha=1e-4, max_iter=300)
poisson.fit(X_train_poi, y_train_rate, sample_weight=exp_train)

# predict rate, then convert to expected claims
pred_rate_train = poisson.predict(X_train_poi)
pred_rate_test  = poisson.predict(X_test_poi)

pred_claims_train = pred_rate_train * exp_train
pred_claims_test  = pred_rate_test * exp_test

# ---------------------------
# evaluate RMSE on ClaimNb
# ---------------------------
rmse_train_poi = np.sqrt(np.mean((y_train_claims - pred_claims_train) ** 2))
rmse_test_poi  = np.sqrt(np.mean((y_test_claims  - pred_claims_test)  ** 2))

print("Poisson (all categoricals) train RMSE:", rmse_train_poi)
print("Poisson (all categoricals) test RMSE:", rmse_test_poi)

# ---------------------------
# look at top coefficients by magnitude for interpretation
# ---------------------------
coef_series = pd.Series(poisson.coef_, index=feature_cols)
coef_abs_sorted = coef_series.reindex(coef_series.abs().sort_values(ascending=False).index)

print("\nTop 15 coefficients by absolute size:")
print(coef_abs_sorted.head(15))


#Same model but with all categorical values encoded

In [18]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import PoissonRegressor

df_train_full = pd.read_csv("data/claims_train_clean.csv")
df_test_full  = pd.read_csv("data/claims_test_clean.csv")

print(df_train_full.shape, df_test_full.shape)
df_train_full.head(3)


(541416, 13) (135373, 13)


,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region,ClaimRate
0,2122523.0,0,0.43,D,7,18,36,95,B1,Regular,1054,R24,0.0
1,3173420.0,0,0.10,D,7,17,80,95,B2,Regular,598,R25,0.0
2,1188619.0,0,0.33,E,7,3,36,76,B6,Regular,4172,R82,0.0


In [19]:
df_train_full = df_train_full.drop(columns=["IDpol", "ClaimRate"])
df_test_full  = df_test_full.drop(columns=["IDpol", "ClaimRate"])

In [20]:
#Expanded cattegorical columns
numeric_features = ["Exposure", "VehPower", "VehAge", "DrivAge", "BonusMalus", "Density"]
categorical_features = ["Area", "VehBrand", "VehGas", "Region"]

In [21]:
#dummies for train/test
train_cat = pd.get_dummies(df_train_full[categorical_features], drop_first=False)
test_cat  = pd.get_dummies(df_test_full[categorical_features], drop_first=False)

#align columns so test has same dummy columns as train
test_cat = test_cat.reindex(columns=train_cat.columns, fill_value=0)

print("train_cat shape:", train_cat.shape)
print("test_cat shape:", test_cat.shape)

train_cat shape: (541416, 41)
test_cat shape: (135373, 41)


In [22]:
#numeric part
X_train_num = df_train_full[numeric_features].copy()
X_test_num  = df_test_full[numeric_features].copy()

#combine numeric and dummies
X_train_full_design = pd.concat([X_train_num.reset_index(drop=True),
                                 train_cat.reset_index(drop=True)], axis=1)

X_test_full_design = pd.concat([X_test_num.reset_index(drop=True),
                                test_cat.reset_index(drop=True)], axis=1)

feature_cols_full = list(X_train_full_design.columns)

print("X_train_full_design shape:", X_train_full_design.shape)
print("X_test_full_design shape:", X_test_full_design.shape)

#scale numeric features only - leave dummies as 0/1
scaler_full = StandardScaler()

X_train_scaled_full = X_train_full_design.copy()
X_test_scaled_full  = X_test_full_design.copy()

X_train_scaled_full[numeric_features] = scaler_full.fit_transform(X_train_full_design[numeric_features])
X_test_scaled_full[numeric_features]  = scaler_full.transform(X_test_full_design[numeric_features])

X_train_poi_full = X_train_scaled_full.values.astype(float)
X_test_poi_full  = X_test_scaled_full.values.astype(float)

print("X_train_poi_full shape:", X_train_poi_full.shape)
print("X_test_poi_full shape:", X_test_poi_full.shape)

X_train_full_design shape: (541416, 47)
X_test_full_design shape: (135373, 47)
X_train_poi_full shape: (541416, 47)
X_test_poi_full shape: (135373, 47)


In [23]:
#target- number of claims
y_train_claims_full = df_train_full["ClaimNb"].values
y_test_claims_full  = df_test_full["ClaimNb"].values

#exposure
exp_train_full = df_train_full["Exposure"].values
exp_test_full  = df_test_full["Exposure"].values

#masks to avoid exposure = 0
train_mask_full = exp_train_full > 0
test_mask_full  = exp_test_full > 0

X_train_poi_use = X_train_poi_full[train_mask_full]
X_test_poi_use  = X_test_poi_full[test_mask_full]

y_train_claims_use = y_train_claims_full[train_mask_full]
y_test_claims_use  = y_test_claims_full[test_mask_full]

exp_train_use = exp_train_full[train_mask_full]
exp_test_use  = exp_test_full[test_mask_full]

#rate = claims per unit exposure
y_train_rate_use = y_train_claims_use / exp_train_use

print("rate min:", y_train_rate_use.min(), "rate max:", y_train_rate_use.max())


rate min: 0.0 rate max: 732.0000000000188


In [24]:
poisson_full = PoissonRegressor(alpha=1e-4, max_iter=300)

poisson_full.fit(X_train_poi_use, y_train_rate_use, sample_weight=exp_train_use)

#predict rate, then convert back to expected number of claims
pred_rate_train_full = poisson_full.predict(X_train_poi_use)
pred_rate_test_full  = poisson_full.predict(X_test_poi_use)

pred_claims_train_full = pred_rate_train_full * exp_train_use
pred_claims_test_full  = pred_rate_test_full * exp_test_use

#RMSE on ClaimNb
rmse_train_poi_full = np.sqrt(np.mean((y_train_claims_use - pred_claims_train_full) ** 2))
rmse_test_poi_full  = np.sqrt(np.mean((y_test_claims_use  - pred_claims_test_full)  ** 2))

print("Poisson (all categoricals) train RMSE:", rmse_train_poi_full)
print("Poisson (all categoricals) test RMSE:", rmse_test_poi_full)

Poisson (all categoricals) train RMSE: 0.23739587016111988
Poisson (all categoricals) test RMSE: 0.24385641270758748


In [25]:
coef_series_full = pd.Series(poisson_full.coef_, index=feature_cols_full)
coef_abs_sorted_full = coef_series_full.reindex(coef_series_full.abs().sort_values(ascending=False).index)

print("\nTop 20 coefficients by absolute value:")
print(coef_abs_sorted_full.head(20))


Top 20 coefficients by absolute value:
Exposure         -0.491458
BonusMalus        0.297554
Region_R83       -0.223666
Region_R73       -0.206549
VehAge           -0.200734
Region_R53        0.190638
Region_R24        0.163546
Region_R74        0.160952
Region_R41       -0.158705
Region_R82        0.145523
DrivAge           0.135450
Region_R23       -0.109272
Region_R31       -0.108553
VehBrand_B5       0.099153
VehBrand_B14     -0.091311
Region_R25        0.089080
VehGas_Regular    0.084564
Area_A           -0.082613
Region_R21        0.078130
VehBrand_B11      0.077150
dtype: float64
